# Motores de Reconocimiento Automatico del Habla (ASR)

Comparacion practica de dos motores de **transcripcion de audio**:

1. **faster-whisper** — ejecucion **local en CPU**, sin costo por uso, control total de los datos.
2. **OpenAI API** — ejecucion en **la nube**, modelos `whisper-1` y `gpt-4o-transcribe`.

Conceptos clave de la clase:

- **ASR** (*Automatic Speech Recognition*): conversion de voz a texto.
- **Transcripcion**: el texto resultante del audio.
- **Diarizacion**: identificacion de *quien habla y cuando* (segmentacion por hablante).
- **Timestamps**: marcas de tiempo por segmento.
- **Temperatura**: controla la aleatoriedad. Usamos `0.2` para resultados **deterministas**.

## 0. Dependencias y configuracion

Instalacion de paquetes (ejecutar una sola vez). Para diarizacion se requiere `pyannote.audio`
y un **token de Hugging Face**; para grabacion en vivo, `sounddevice`.

In [ ]:
%pip install -q faster-whisper openai python-dotenv pyannote.audio sounddevice scipy torch

### Variables de entorno (`.env`)

Crear un archivo `.env` en el directorio del notebook con las **credenciales**:

```
OPENAI_API_KEY=sk-...
HUGGINGFACE_TOKEN=hf_...
```

El token de Hugging Face es **opcional**: solo se usa para la diarizacion con `pyannote`.

In [ ]:
import os
import time
import urllib.request
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

# Parametros globales de la clase
SAMPLE_RATE = 16000   # 16 kHz: frecuencia recomendada para Whisper
TEMPERATURE = 0.2     # baja aleatoriedad -> transcripcion mas determinista


def fmt_ts(seconds: float) -> str:
    """Formatea segundos como mm:ss."""
    minutes, secs = divmod(int(round(seconds)), 60)
    return f"{minutes:02d}:{secs:02d}"


print("Entorno cargado. SAMPLE_RATE =", SAMPLE_RATE, "| TEMPERATURE =", TEMPERATURE)

### Audio de trabajo

Se define la **ruta del audio**. Si no existe localmente, se descarga una muestra en ingles
para que la clase sea reproducible sin archivos previos.

In [ ]:
AUDIO_PATH = Path("materials/audio2.ogg")   # <-- ajustar a la ruta local

SAMPLE_URL = "https://www.voiptroubleshooter.com/open_speech/american/OSR_us_000_0010_8k.wav"
SAMPLE_FILE = Path("sample_english.wav")


def ensure_audio(path: Path) -> Path:
    """Devuelve la ruta del audio; descarga una muestra si no existe."""
    if path.exists():
        return path
    print(f"No se encontro {path}. Descargando muestra de ejemplo...")
    urllib.request.urlretrieve(SAMPLE_URL, SAMPLE_FILE)
    print(f"Muestra descargada: {SAMPLE_FILE}")
    return SAMPLE_FILE


audio_file = ensure_audio(AUDIO_PATH)
print(f"Audio en uso: {audio_file}")

---
## 1. faster-whisper (local, CPU)

Motor **offline** basado en Whisper optimizado con `CTranslate2`. Ventajas: **privacidad**
de los datos (nada sale del equipo), **sin costo por uso** y control del tamano del modelo.

### 1.1 Configuracion y carga del modelo

- `device="cpu"` fuerza ejecucion en **CPU**.
- `compute_type="int8"` aplica **cuantizacion** int8, optima en CPU.
- `MODEL_SIZE` controla precision vs. velocidad: `small` es agil para clase;
  `large-v3` es mas preciso pero **mas lento** en CPU.

In [ ]:
from faster_whisper import WhisperModel

DEVICE = "cpu"
COMPUTE_TYPE = "int8"      # cuantizacion int8: optima para CPU
MODEL_SIZE = "small"       # alternativas: base, medium, large-v3 (mayor precision, mas lento)

print(f"Cargando modelo faster-whisper '{MODEL_SIZE}' en {DEVICE}...")
fw_model = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE, cpu_threads=8)
print("Modelo cargado.")

### 1.2 Transcripcion del archivo

`transcribe` devuelve un **generador** de segmentos. Lo materializamos con `list(...)`
para poder **reutilizarlo** despues (por ejemplo, en la alineacion con la diarizacion).

In [ ]:
start = time.time()
segments_gen, info = fw_model.transcribe(
    str(audio_file),
    beam_size=5,
    language="en",          # "es" para espanol; None para deteccion automatica
    temperature=TEMPERATURE,
)
segments = list(segments_gen)   # materializar el generador para reutilizarlo
elapsed = time.time() - start

print(f"Idioma detectado: {info.language} (probabilidad: {info.language_probability:.2f})")
print(f"Tiempo de transcripcion: {elapsed:.2f} s")
print("\nTranscripcion:")
for seg in segments:
    print(f"[{fmt_ts(seg.start)} -> {fmt_ts(seg.end)}] {seg.text.strip()}")

### 1.3 Diarizacion con pyannote

La **diarizacion** identifica los turnos de cada **hablante** (`SPEAKER_00`, `SPEAKER_01`, ...).
Requiere `HUGGINGFACE_TOKEN`. Si no esta configurado, el bloque se omite con elegancia.

In [ ]:
import torch
from pyannote.audio import Pipeline

hf_token = os.getenv("HUGGINGFACE_TOKEN")
diarization = None

if not hf_token:
    print("HUGGINGFACE_TOKEN no configurado en .env. Se omite la diarizacion.")
else:
    print("Cargando pipeline de diarizacion (pyannote)...")
    diar_pipeline = Pipeline.from_pretrained(
        "pyannote/speaker-diarization-3.1",
        use_auth_token=hf_token,
    )
    diar_pipeline.to(torch.device("cpu"))   # forzar CPU

    print("Ejecutando diarizacion...")
    diarization = diar_pipeline(str(audio_file))

    print("\nSegmentos por hablante:")
    for turn, _, speaker in diarization.itertracks(yield_label=True):
        print(f"[{fmt_ts(turn.start)} -> {fmt_ts(turn.end)}] {speaker}")

### 1.4 Alineacion: transcripcion + hablante

Mejora practica: combinamos **que se dijo** (transcripcion) con **quien lo dijo** (diarizacion).
Para cada segmento de texto buscamos el hablante activo en su **punto medio**.

In [ ]:
def speaker_at(diar, t: float) -> str:
    """Hablante activo en el instante t (segundos)."""
    if diar is None:
        return "N/A"
    for turn, _, speaker in diar.itertracks(yield_label=True):
        if turn.start <= t <= turn.end:
            return speaker
    return "DESCONOCIDO"


if diarization is not None:
    print("Transcripcion con hablante asignado:\n")
    for seg in segments:
        mid = (seg.start + seg.end) / 2
        speaker = speaker_at(diarization, mid)
        print(f"[{fmt_ts(seg.start)} -> {fmt_ts(seg.end)}] {speaker}: {seg.text.strip()}")
else:
    print("Sin diarizacion disponible; se muestra solo la transcripcion (celda 1.2).")

### 1.5 Grabacion en vivo y transcripcion

Capturamos audio del **microfono** con `sounddevice`, lo guardamos como WAV temporal
y lo transcribimos. La funcion `record_audio` se **reutiliza** en la Parte 2.

In [ ]:
import sounddevice as sd
import numpy as np
import scipy.io.wavfile as wav


def record_audio(duration: int = 5, sample_rate: int = SAMPLE_RATE) -> Path:
    """Graba audio del microfono y lo guarda como WAV."""
    print(f"Grabando {duration} segundos...")
    audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1, dtype="float32")
    sd.wait()
    out = Path("temp_recording.wav")
    wav.write(out, sample_rate, (audio * 32767).astype(np.int16))   # convertir a int16
    print("Grabacion completada.")
    return out


rec_path = record_audio(duration=5)

segments_live, info_live = fw_model.transcribe(
    str(rec_path), language="es", temperature=TEMPERATURE
)
print(f"\nIdioma detectado: {info_live.language}")
print("Transcripcion:")
for seg in segments_live:
    print(f"[{fmt_ts(seg.start)} -> {fmt_ts(seg.end)}] {seg.text.strip()}")

rec_path.unlink(missing_ok=True)   # limpiar temporal

---
## 2. OpenAI API (nube)

Motor **gestionado** por OpenAI. Ventajas: **sin instalacion de modelos**, alta calidad
y rapidez. Limitaciones: **costo por uso** y los datos viajan a la API.

### 2.1 Cliente

Inicializamos el cliente con `OPENAI_API_KEY`. Si falta, detenemos con un mensaje claro.

In [ ]:
from openai import OpenAI

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("Configura OPENAI_API_KEY en el archivo .env")

client = OpenAI(api_key=api_key)
print("Cliente OpenAI inicializado.")

### 2.2 Transcripcion con `whisper-1`

Modelo estandar, rapido y estable. Usamos `response_format="verbose_json"` para obtener
**segmentos con timestamps** ademas del texto.

In [ ]:
start = time.time()
with open(audio_file, "rb") as f:
    result = client.audio.transcriptions.create(
        model="whisper-1",
        file=f,
        language="en",                  # "es" para espanol
        response_format="verbose_json", # incluye segmentos con timestamps
        temperature=TEMPERATURE,
    )
elapsed = time.time() - start

print(f"Tiempo: {elapsed:.2f} s")
print(f"\nTexto:\n{result.text}\n")
print("Segmentos:")
for seg in result.segments:
    print(f"[{fmt_ts(seg.start)} -> {fmt_ts(seg.end)}] {seg.text.strip()}")

### 2.3 Transcripcion con `gpt-4o-transcribe`

Modelo de **mayor calidad**. Nota tecnica: solo admite `response_format` en `json` o `text`
(no `verbose_json`). Listamos los modelos disponibles para ajustar el **id exacto** de la cuenta.

In [ ]:
# Verificar modelos de transcripcion disponibles en la cuenta
try:
    available = [m.id for m in client.models.list() if "transcribe" in m.id]
    print(f"Modelos de transcripcion disponibles: {available}")
except Exception as e:
    print(f"No se pudo listar modelos: {e}")

TRANSCRIBE_MODEL = "gpt-4o-transcribe"   # ajustar al id exacto de la cuenta si difiere

try:
    start = time.time()
    with open(audio_file, "rb") as f:
        result_4o = client.audio.transcriptions.create(
            model=TRANSCRIBE_MODEL,
            file=f,
            language="en",
            response_format="json",      # gpt-4o-transcribe no admite verbose_json
            temperature=TEMPERATURE,
        )
    elapsed = time.time() - start
    print(f"\n{TRANSCRIBE_MODEL} (tiempo: {elapsed:.2f} s):")
    print(result_4o.text)
except Exception as e:
    print(f"No se pudo usar {TRANSCRIBE_MODEL}: {e}")
    print("Alternativa: usar whisper-1 (celda 2.2).")

### 2.4 Grabacion en vivo y transcripcion

Reutilizamos `record_audio` de la celda 1.5 para capturar voz y transcribirla con `whisper-1`.

In [ ]:
rec_path = record_audio(duration=5)

with open(rec_path, "rb") as f:
    transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=f,
        language="es",            # "en" si el audio es en ingles
        response_format="text",
        temperature=TEMPERATURE,
    )

print(f"\nTranscripcion:\n{transcript}")

rec_path.unlink(missing_ok=True)

### 2.5 Traduccion del texto transcrito

Mejora: en lugar de `input()` (que bloquea la ejecucion), usamos una variable `TARGET_LANG`
**reproducible**. Se traduce con `gpt-4o-mini` y `temperature=0.2`.

In [ ]:
TARGET_LANG = "en"   # idioma destino; "" para omitir. Ej.: "en", "fr", "de"

if TARGET_LANG:
    translation = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=TEMPERATURE,
        messages=[
            {"role": "system",
             "content": f"Traduce el texto al idioma '{TARGET_LANG}'. Devuelve unicamente la traduccion."},
            {"role": "user", "content": transcript},
        ],
    )
    print(f"Traduccion ({TARGET_LANG}): {translation.choices[0].message.content}")
else:
    print("Traduccion omitida.")

---
## 3. Comparacion de los dos motores

| Criterio            | faster-whisper (local)        | OpenAI API (nube)             |
|---------------------|-------------------------------|-------------------------------|
| **Ejecucion**       | CPU/GPU propia                | Servidores de OpenAI          |
| **Costo**           | Sin costo por uso             | Costo por minuto/token        |
| **Privacidad**      | Datos no salen del equipo     | Datos enviados a la API       |
| **Diarizacion**     | Si (con `pyannote`)           | No nativa                     |
| **Velocidad CPU**   | Depende del `MODEL_SIZE`      | Generalmente rapida           |
| **Instalacion**     | Modelos locales               | Solo `API key`                |

**Conclusion para clase**: faster-whisper es ideal cuando importan la **privacidad** y el
**costo cero**; OpenAI API conviene cuando se prioriza **simplicidad** y **calidad** sin
gestionar infraestructura.